# FashionNet - CNN Image Classification
## Fashion MNIST Dataset
Classifying 10 fashion categories using Convolutional Neural Networks

## 01 — Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, confusion_matrix

print('TensorFlow version:', tf.__version__)

CLASS_NAMES = ['T-Shirt','Trouser','Pullover','Dress','Coat',
               'Sandal','Shirt','Sneaker','Bag','Ankle Boot']
CLASS_ICONS = ['👕','👖','🧥','👗','🧣','👡','👔','👟','👜','👢']

## 02 — Load & Preprocess Data

In [ ]:
(X_train_full, y_train_full), (X_test_full, y_test_full) = fashion_mnist.load_data()

# Use subset for speed
X_train = X_train_full[:2500].astype('float32') / 255.0
X_test  = X_test_full[:500].astype('float32')   / 255.0
y_train = y_train_full[:2500]
y_test  = y_test_full[:500]

# Add channel dimension (28,28) -> (28,28,1)
X_train = X_train[..., np.newaxis]
X_test  = X_test[..., np.newaxis]

y_train_cat = to_categorical(y_train, 10)
y_test_cat  = to_categorical(y_test,  10)

print('Train shape :', X_train.shape)
print('Test shape  :', X_test.shape)
print('Classes     :', CLASS_NAMES)

## 03 — Exploratory Data Analysis

In [ ]:
# Class distribution
unique, counts = np.unique(y_train, return_counts=True)
plt.figure(figsize=(12, 4))
bars = plt.bar([CLASS_NAMES[i] for i in unique], counts, color='#ff3d00', edgecolor='none')
plt.title('Class Distribution in Training Set', fontsize=13, fontweight='bold')
plt.xticks(rotation=30, ha='right')
plt.ylabel('Count')
for bar, count in zip(bars, counts):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             str(count), ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Sample images from each class
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
fig.patch.set_facecolor('#1a1a1a')
for i in range(10):
    ax = axes[i//5][i%5]
    idx = np.where(y_train == i)[0][0]
    ax.imshow(X_train[idx].squeeze(), cmap='gray')
    ax.set_title(f'{CLASS_ICONS[i]} {CLASS_NAMES[i]}', color='white', fontsize=9, fontweight='bold')
    ax.axis('off')
    ax.set_facecolor('#1a1a1a')
plt.suptitle('One Sample Per Class', color='white', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Pixel intensity distribution
plt.figure(figsize=(10, 4))
plt.hist(X_train.flatten(), bins=50, color='#ff3d00', alpha=0.8, edgecolor='none')
plt.title('Pixel Intensity Distribution', fontsize=13, fontweight='bold')
plt.xlabel('Pixel Value (normalized)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

## 04 — Build CNN Model

In [ ]:
model = Sequential([
    Conv2D(16, (3,3), activation='relu', padding='same', input_shape=(28,28,1)),
    MaxPooling2D(2,2),
    Dropout(0.2),

    Conv2D(32, (3,3), activation='relu', padding='same'),
    MaxPooling2D(2,2),
    Dropout(0.2),

    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(10, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 05 — Train Model

In [ ]:
early_stop = EarlyStopping(patience=2, restore_best_weights=True)

history = model.fit(
    X_train, y_train_cat,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

print('\nTraining complete!')

## 06 — Training History

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
fig.patch.set_facecolor('#f5f0e8')

for ax in [ax1, ax2]:
    ax.set_facecolor('#1a1a1a')
    for s in ax.spines.values(): s.set_color('#333')
    ax.tick_params(colors='#666')

epochs_range = range(len(history.history['accuracy']))

ax1.plot(epochs_range, history.history['accuracy'],     color='#ff3d00', linewidth=2.5, label='Train', marker='o')
ax1.plot(epochs_range, history.history['val_accuracy'], color='#f5f0e8', linewidth=2.5, label='Val',   linestyle='--', marker='s')
ax1.fill_between(epochs_range, history.history['accuracy'], alpha=0.1, color='#ff3d00')
ax1.set_title('ACCURACY', color='#f5f0e8', fontsize=11, fontweight='bold')
ax1.legend(facecolor='#111', labelcolor='#999')
ax1.set_xlabel('Epoch', color='#666')

ax2.plot(epochs_range, history.history['loss'],     color='#ff3d00', linewidth=2.5, label='Train', marker='o')
ax2.plot(epochs_range, history.history['val_loss'], color='#f5f0e8', linewidth=2.5, label='Val',   linestyle='--', marker='s')
ax2.fill_between(epochs_range, history.history['loss'], alpha=0.1, color='#ff3d00')
ax2.set_title('LOSS', color='#f5f0e8', fontsize=11, fontweight='bold')
ax2.legend(facecolor='#111', labelcolor='#999')
ax2.set_xlabel('Epoch', color='#666')

plt.tight_layout()
plt.show()

## 07 — Evaluate Model

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test_cat, verbose=0)
print(f'Test Loss     : {loss:.4f}')
print(f'Test Accuracy : {accuracy:.2%}')

y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
y_true = y_test.flatten()

print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

## 08 — Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
fig.patch.set_facecolor('#f5f0e8')
ax.set_facecolor('#1a1a1a')

cm = confusion_matrix(y_true, y_pred)
import matplotlib.colors as mcolors
cmap = mcolors.LinearSegmentedColormap.from_list('rr', ['#111', '#ff3d00'])

sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, ax=ax,
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=1, linecolor='#1a1a1a',
            annot_kws={'size': 9, 'color': 'white', 'weight': 'bold'})

ax.set_xlabel('Predicted', color='#888', fontsize=10)
ax.set_ylabel('Actual',    color='#888', fontsize=10)
ax.set_title('Confusion Matrix', color='#f5f0e8', fontsize=13, fontweight='bold', pad=14)
ax.tick_params(colors='#888')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 09 — Per-Class Accuracy

In [ ]:
class_acc = []
for i in range(10):
    mask = y_true == i
    acc  = np.mean(y_pred[mask] == i) if mask.sum() > 0 else 0
    class_acc.append(acc)

fig, ax = plt.subplots(figsize=(13, 4))
fig.patch.set_facecolor('#f5f0e8')
ax.set_facecolor('#1a1a1a')

bar_colors = ['#ff3d00' if a >= 0.7 else '#ff3d0066' for a in class_acc]
bars = ax.bar(
    [f"{CLASS_ICONS[i]} {CLASS_NAMES[i]}" for i in range(10)],
    class_acc, color=bar_colors, edgecolor='none', width=0.6
)
ax.axhline(np.mean(class_acc), color='#f5f0e8', linestyle='--', linewidth=1.5,
           label=f'Mean: {np.mean(class_acc):.2%}')
ax.set_ylabel('Accuracy', color='#666')
ax.set_ylim(0, 1.15)
ax.legend(facecolor='#1a1a1a', labelcolor='#999')
ax.set_title('Per-Class Accuracy', color='#f5f0e8', fontsize=13, fontweight='bold')
for s in ax.spines.values(): s.set_color('#333')
ax.tick_params(colors='#888', labelsize=8)
plt.xticks(rotation=20, ha='right', color='#aaa')

for bar, acc in zip(bars, class_acc):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{acc:.0%}', ha='center', va='bottom', color='#f5f0e8',
            fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

## 10 — Sample Predictions Grid

In [ ]:
indices = np.random.choice(len(X_test), 15, replace=False)
fig, axes = plt.subplots(3, 5, figsize=(12, 7))
fig.patch.set_facecolor('#1a1a1a')

for i, idx in enumerate(indices):
    ax = axes[i//5][i%5]
    ax.imshow(X_test[idx].squeeze(), cmap='gray')
    pred  = CLASS_NAMES[y_pred[idx]]
    true  = CLASS_NAMES[y_true[idx]]
    color = '#00e676' if pred == true else '#ff3d00'
    ax.set_title(f'{CLASS_ICONS[y_pred[idx]]} {pred}\n({true})',
                 color=color, fontsize=7, fontweight='bold')
    ax.axis('off')

plt.suptitle('Sample Predictions  (green=correct, red=wrong)',
             color='white', fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

## 11 — Save Model

In [ ]:
model.save('fashionnet_model.h5')
print('✅ Model saved as fashionnet_model.h5')